<a href="https://colab.research.google.com/github/NAMUORI00/aerospace-rag/blob/main/notebooks/aerospace_rag_colab_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 항공우주 로컬 RAG Colab 노트북

이 노트북은 `smartfarm-workspace`에서 계승한 Dense/Sparse/Graph RAG 구조를 항공우주 문서용 Google Colab T4 데모로 정리한 학습형 실행 문서입니다. Colab을 실행할 때마다 GitHub에서 프로젝트를 새로 clone하고, `/content/aerospace-rag/data`에 둔 업무 파일을 읽어 로컬 인덱스를 만든 뒤, 근거가 표시되는 답변까지 확인합니다.

**목표**
- 기업 데모용: 항공우주 문서를 Colab T4에 올리고 RAG 답변, 상위 근거, 검색 진단을 한 화면에서 검토합니다.
- 학습용 구조 이해: chunk, vector store, BM25 문서, graph-lite entity/relation이 어떤 지식 데이터베이스 구조로 저장되는지 확인합니다.
- 검증용: 최소 세 개 이상의 업무형 질문에 대해 답변, source, retrieval diagnostics가 함께 남는지 확인합니다.

**실행 순서**
1. Colab 런타임을 GPU T4로 설정합니다.
2. 위에서 아래로 모든 셀을 순서대로 실행합니다.
3. 데이터 준비 섹션에서 지원 파일을 `/content/aerospace-rag/data` 아래에 둡니다.
4. 인덱스 생성 섹션에서 `qdrant`, `graph_index.json`, `bm25.json`, `chunks.jsonl` 산출물이 만들어졌는지 확인합니다.
5. 8A 섹션에서 도메인 데이터베이스 저장 구조와 지식 그래프 노드를 확인한 뒤 검색, 답변, 근거, 반복 질문 결과를 검토합니다.

**문제 대응**
- 패키지 import가 실패하면 의존성 설치 셀을 다시 실행합니다.
- vLLM 모델 로딩이 실패하면 5번 런타임 셀을 다시 실행합니다.
- 데이터가 없다고 나오면 파일 위치가 `/content/aerospace-rag/data`인지 확인합니다.
- LLM 생성 없이 retrieval만 점검할 때는 4번 설정 셀에서 `ANSWER_PROVIDER = "extractive"`, `EXTRACTOR_LLM_BACKEND = "local_fallback"`로 바꿉니다.

## 1. 실행 환경 확인

현재 런타임이 Colab인지 로컬인지, Python/OS/GPU 상태가 무엇인지 먼저 기록합니다. 이 정보는 같은 노트북을 다시 실행했을 때 환경 차이를 비교하는 기준입니다.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from IPython.display import HTML, Markdown, display

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def current_working_dir() -> Path:
    try:
        return Path.cwd()
    except FileNotFoundError:
        cwd_target = Path("/content") if Path("/content").exists() else Path.home()
        os.chdir(cwd_target)
        return Path.cwd()


RUN_STARTED_AT = time.strftime("%Y-%m-%d %H:%M:%S %Z", time.localtime())
print("Run started:", RUN_STARTED_AT)
print("Runtime:", "Colab" if IN_COLAB else "Local")
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()
except Exception:
    gpu_info = "No GPU reported by nvidia-smi"
print("GPU:", gpu_info)
print("Initial cwd:", current_working_dir())

## 2. 프로젝트 소스 확보

Colab에서는 항상 `/content`로 이동한 뒤 기존 `/content/aerospace-rag`를 삭제하고 GitHub repo를 새로 clone합니다. clone 후 commit hash를 출력해 이번 실행이 어떤 코드 상태였는지 남깁니다.

In [ ]:
GITHUB_REPO_URL = "https://github.com/NAMUORI00/aerospace-rag.git"
DEFAULT_COLAB_ROOT = Path("/content/aerospace-rag")

if IN_COLAB:
    os.chdir(DEFAULT_COLAB_ROOT.parent)
    print("Policy: Project source is cloned fresh for each Colab run.")
    print("Running git clone:", GITHUB_REPO_URL)
    if DEFAULT_COLAB_ROOT.exists():
        shutil.rmtree(DEFAULT_COLAB_ROOT)
    subprocess.check_call(["git", "clone", GITHUB_REPO_URL, str(DEFAULT_COLAB_ROOT)])
    os.chdir(DEFAULT_COLAB_ROOT)
else:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

from aerospace_rag.notebook_runtime import ensure_valid_cwd, git_output

PROJECT_ROOT = ensure_valid_cwd(DEFAULT_COLAB_ROOT, GITHUB_REPO_URL, in_colab=False)
REPO_BRANCH = git_output("rev-parse", "--abbrev-ref", "HEAD")
REPO_COMMIT = git_output("rev-parse", "HEAD")
print("Project root:", PROJECT_ROOT)
print("In Colab:", IN_COLAB)
print("Repo URL:", GITHUB_REPO_URL)
print("Git branch:", REPO_BRANCH)
print("Git commit:", REPO_COMMIT)

## 3. 의존성 설치와 버전 고정 확인

필요한 Python 패키지를 설치하고, 재현성 비교에 필요한 핵심 패키지 버전을 출력합니다.

In [ ]:
from aerospace_rag.notebook_runtime import ensure_dependencies

PACKAGE_SNAPSHOT = ensure_dependencies(PROJECT_ROOT, IN_COLAB)

## 4. 실행 설정 확정

이번 실행에서 사용할 RAG 설정을 한곳에 모읍니다. 아래 블록이 사용자가 직접 수정하는 기본 설정입니다. 모델명, 답변 provider, embedding backend, vector backend, 검색 개수, index reset 여부를 이 셀에서 먼저 확정하고 이후 모든 단계가 같은 값을 사용합니다.

In [ ]:
# User-editable defaults
DATA_DIR = PROJECT_ROOT / "data"
INDEX_DIR = DATA_DIR / "index"

ANSWER_PROVIDER = "vllm"  # "vllm" or "extractive"
TOP_K = 5

VLLM_MODEL = "google/gemma-4-E4B-it"
VLLM_DTYPE = "auto"
VLLM_QUANTIZATION = ""
VLLM_LOAD_FORMAT = "auto"
VLLM_GPU_MEMORY_UTILIZATION = 0.82
VLLM_MAX_MODEL_LEN = 2048
VLLM_CPU_OFFLOAD_GB = 4.0
VLLM_TRUST_REMOTE_CODE = False
VLLM_ENFORCE_EAGER = True
LLM_ANSWER_MAX_TOKENS = 1024
LLM_EXTRACT_MAX_TOKENS = 768

EXTRACTOR_LLM_BACKEND = "vllm"  # "vllm" or explicit debug mode "local_fallback"
KNOWLEDGE_EXTRACT_RETRIES = 1
KNOWLEDGE_EXTRACT_REPAIR_RETRIES = 1
KNOWLEDGE_EXTRACT_MAX_CHARS = 1200

EMBED_BACKEND = "hash"
EMBED_MODEL = "hash"
VECTOR_BACKEND = "qdrant"  # explicit debug mode: "json"
INDEX_RESET = True

SUPPORTED_ANSWER_PROVIDERS = {"vllm", "extractive"}
if ANSWER_PROVIDER not in SUPPORTED_ANSWER_PROVIDERS:
    raise ValueError(f"ANSWER_PROVIDER must be one of {sorted(SUPPORTED_ANSWER_PROVIDERS)}")

SUPPORTED_EXTRACTOR_BACKENDS = {"vllm", "local_fallback"}
if EXTRACTOR_LLM_BACKEND not in SUPPORTED_EXTRACTOR_BACKENDS:
    raise ValueError(f"EXTRACTOR_LLM_BACKEND must be one of {sorted(SUPPORTED_EXTRACTOR_BACKENDS)}")

for legacy_name in (
    "VLLM_MODEL",
    "VLLM_DTYPE",
    "VLLM_GPU_MEMORY_UTILIZATION",
    "VLLM_MAX_MODEL_LEN",
    "VLLM_CPU_OFFLOAD_GB",
    "VLLM_TRUST_REMOTE_CODE",
    "VLLM_ENFORCE_EAGER",
):
    os.environ.pop(legacy_name, None)

os.environ["LLM_PROVIDER"] = ANSWER_PROVIDER
os.environ["AEROSPACE_EMBED_BACKEND"] = EMBED_BACKEND
os.environ["AEROSPACE_EMBED_MODEL"] = EMBED_MODEL
os.environ["AEROSPACE_VECTOR_BACKEND"] = VECTOR_BACKEND
os.environ["AEROSPACE_LLM_MODEL"] = VLLM_MODEL
os.environ["AEROSPACE_VLLM_DTYPE"] = VLLM_DTYPE
os.environ["AEROSPACE_VLLM_QUANTIZATION"] = VLLM_QUANTIZATION
os.environ["AEROSPACE_VLLM_LOAD_FORMAT"] = VLLM_LOAD_FORMAT
os.environ["AEROSPACE_VLLM_GPU_MEMORY_UTILIZATION"] = str(VLLM_GPU_MEMORY_UTILIZATION)
os.environ["AEROSPACE_VLLM_MAX_MODEL_LEN"] = str(VLLM_MAX_MODEL_LEN)
os.environ["AEROSPACE_VLLM_CPU_OFFLOAD_GB"] = str(VLLM_CPU_OFFLOAD_GB)
os.environ["AEROSPACE_VLLM_TRUST_REMOTE_CODE"] = str(VLLM_TRUST_REMOTE_CODE).lower()
os.environ["AEROSPACE_VLLM_ENFORCE_EAGER"] = str(VLLM_ENFORCE_EAGER).lower()
os.environ["LLM_ANSWER_MAX_TOKENS"] = str(LLM_ANSWER_MAX_TOKENS)
os.environ["LLM_EXTRACT_MAX_TOKENS"] = str(LLM_EXTRACT_MAX_TOKENS)
os.environ["EXTRACTOR_LLM_BACKEND"] = EXTRACTOR_LLM_BACKEND
os.environ["KNOWLEDGE_EXTRACT_RETRIES"] = str(KNOWLEDGE_EXTRACT_RETRIES)
os.environ["KNOWLEDGE_EXTRACT_REPAIR_RETRIES"] = str(KNOWLEDGE_EXTRACT_REPAIR_RETRIES)
os.environ["KNOWLEDGE_EXTRACT_MAX_CHARS"] = str(KNOWLEDGE_EXTRACT_MAX_CHARS)

LLM_NEEDED = ANSWER_PROVIDER == "vllm" or EXTRACTOR_LLM_BACKEND == "vllm"

RUNTIME_CONFIG = {
    "DATA_DIR": str(DATA_DIR),
    "INDEX_DIR": str(INDEX_DIR),
    "ANSWER_PROVIDER": ANSWER_PROVIDER,
    "TOP_K": TOP_K,
    "AEROSPACE_LLM_MODEL": VLLM_MODEL,
    "AEROSPACE_VLLM_DTYPE": VLLM_DTYPE,
    "AEROSPACE_VLLM_QUANTIZATION": VLLM_QUANTIZATION,
    "AEROSPACE_VLLM_LOAD_FORMAT": VLLM_LOAD_FORMAT,
    "AEROSPACE_VLLM_GPU_MEMORY_UTILIZATION": VLLM_GPU_MEMORY_UTILIZATION,
    "AEROSPACE_VLLM_MAX_MODEL_LEN": VLLM_MAX_MODEL_LEN,
    "AEROSPACE_VLLM_CPU_OFFLOAD_GB": VLLM_CPU_OFFLOAD_GB,
    "AEROSPACE_VLLM_TRUST_REMOTE_CODE": VLLM_TRUST_REMOTE_CODE,
    "AEROSPACE_VLLM_ENFORCE_EAGER": VLLM_ENFORCE_EAGER,
    "LLM_ANSWER_MAX_TOKENS": LLM_ANSWER_MAX_TOKENS,
    "LLM_EXTRACT_MAX_TOKENS": LLM_EXTRACT_MAX_TOKENS,
    "EXTRACTOR_LLM_BACKEND": EXTRACTOR_LLM_BACKEND,
    "KNOWLEDGE_EXTRACT_RETRIES": KNOWLEDGE_EXTRACT_RETRIES,
    "KNOWLEDGE_EXTRACT_REPAIR_RETRIES": KNOWLEDGE_EXTRACT_REPAIR_RETRIES,
    "KNOWLEDGE_EXTRACT_MAX_CHARS": KNOWLEDGE_EXTRACT_MAX_CHARS,
    "AEROSPACE_EMBED_BACKEND": EMBED_BACKEND,
    "AEROSPACE_EMBED_MODEL": EMBED_MODEL,
    "AEROSPACE_VECTOR_BACKEND": VECTOR_BACKEND,
    "INDEX_RESET": INDEX_RESET,
    "LLM_NEEDED": LLM_NEEDED,
}
print(json.dumps(RUNTIME_CONFIG, ensure_ascii=False, indent=2))

## 5. vLLM 런타임과 모델 준비

답변 생성과 graph extraction 모두 vLLM provider로 `google/gemma-4-E4B-it`을 직접 로딩합니다. Colab T4에서는 같은 HF 모델을 유지하되 앱 전용 환경변수 `AEROSPACE_VLLM_DTYPE = "auto"`, `AEROSPACE_VLLM_LOAD_FORMAT = "auto"`, `AEROSPACE_VLLM_GPU_MEMORY_UTILIZATION = 0.82`, `AEROSPACE_VLLM_MAX_MODEL_LEN = 2048`, `AEROSPACE_VLLM_CPU_OFFLOAD_GB = 4.0`, `AEROSPACE_VLLM_ENFORCE_EAGER = True` 설정으로 일부 weight를 CPU RAM에 offload해서 T4 메모리에 맞게 모델을 올립니다. 로딩이나 생성이 실패하면 4번 설정 셀의 `VLLM_MODEL`, `AEROSPACE_VLLM_*`, `LLM_*_MAX_TOKENS` 값을 조정한 뒤 이 셀부터 다시 실행하세요.

In [ ]:
from aerospace_rag.notebook_runtime import ensure_model_runtime

LLM_STATUS = ensure_model_runtime(LLM_NEEDED)
if LLM_NEEDED and not LLM_STATUS.get("ready"):
    raise RuntimeError(f"LLM runtime is not ready: {LLM_STATUS.get('reason')}")

## 6. 데이터 파일 준비

`data/` 폴더를 재귀적으로 훑어 지원 형식의 문서를 자동으로 찾습니다. 특정 파일명을 강제하지 않습니다. Colab에서는 파일 패널의 `/content/aerospace-rag/data` 폴더에 업무 문서를 넣은 뒤 이 셀부터 다시 실행하면 됩니다. 파일이 준비되면 파일 크기와 SHA-256 해시를 출력해 같은 데이터로 재현했는지 확인합니다.

In [ ]:
from aerospace_rag.ingestion import SUPPORTED_SUFFIXES
from aerospace_rag.notebook_runtime import discover_data_files

DATA_MANIFEST = discover_data_files(DATA_DIR)
print("데이터 경로:", DATA_DIR)
print("지원 형식:", ", ".join(sorted(SUPPORTED_SUFFIXES)))
if not DATA_MANIFEST:
    print("위 경로에 지원 문서 파일을 넣은 뒤 이 셀을 다시 실행하세요.")
    print("Colab 파일 패널에서 aerospace-rag/data 폴더로 파일을 드래그해 넣으면 됩니다.")
    raise FileNotFoundError(f"지원되는 data 파일이 없습니다: {DATA_DIR}")

for entry in DATA_MANIFEST:
    print(f"OK {entry['name']} bytes={entry['bytes']} sha256={entry['sha256'][:16]}...")
print("Data files:", len(DATA_MANIFEST))

## 7. 수집/파싱 단독 확인

인덱스를 만들기 전에 입력 파일이 어떤 chunk로 변환되는지 확인합니다. 이 단계가 통과하면 데이터 파일명, 파일 형식, 기본 파서가 정상이라는 뜻입니다.

In [ ]:
from collections import Counter
from aerospace_rag.ingestion import ingest_data

chunks = ingest_data(DATA_DIR, strict_expected=False)
CHUNK_SUMMARY = {
    "total_chunks": len(chunks),
    "by_file": dict(Counter(chunk.source_file for chunk in chunks)),
    "by_modality": dict(Counter(chunk.modality for chunk in chunks)),
}
print(json.dumps(CHUNK_SUMMARY, ensure_ascii=False, indent=2))

for chunk in chunks[:5]:
    loc = f"p.{chunk.page}" if chunk.page else f"{chunk.sheet}:{chunk.row}" if chunk.row else "table"
    preview = chunk.text.replace("\n", " ")[:240]
    print(f"- {chunk.chunk_id} [{chunk.source_file} / {loc} / {chunk.modality}]")
    print(" ", preview)

## 8. 인덱스 생성

동일 데이터에서 Qdrant local mode, BM25, graph-lite helper index를 다시 생성합니다. `reset=True`라 이전 index 산출물은 재사용하지 않습니다.

In [ ]:
from aerospace_rag.pipeline import build_index

build_started = time.perf_counter()
result = build_index(data_dir=DATA_DIR, index_dir=INDEX_DIR, reset=INDEX_RESET, strict_expected=False)
INDEX_BUILD_SECONDS = round(time.perf_counter() - build_started, 3)

INDEX_ARTIFACTS = {
    "qdrant": INDEX_DIR / "qdrant",
    "graph_index": result.graph_index_path,
    "bm25": result.bm25_path,
    "chunks": result.chunks_path,
}
ARTIFACT_REPORT = {
    name: {
        "path": str(path),
        "exists": path.exists(),
        "files": sum(1 for _ in path.rglob("*")) if path.is_dir() else 1 if path.exists() else 0,
    }
    for name, path in INDEX_ARTIFACTS.items()
}
missing_artifacts = [name for name, info in ARTIFACT_REPORT.items() if not info["exists"]]
if missing_artifacts:
    raise FileNotFoundError(f"Missing index artifacts: {missing_artifacts}")

BUILD_REPORT = {
    "data_dir": str(result.data_dir),
    "index_dir": str(result.index_dir),
    "file_count": result.file_count,
    "chunk_count": result.chunk_count,
    "qdrant_collection": result.qdrant_collection,
    "graph_index_path": str(result.graph_index_path),
    "bm25_path": str(result.bm25_path),
    "chunks_path": str(result.chunks_path),
    "artifacts": ARTIFACT_REPORT,
    "reset": INDEX_RESET,
    "seconds": INDEX_BUILD_SECONDS,
}
print(json.dumps(BUILD_REPORT, ensure_ascii=False, indent=2))

## 8A. 도메인 데이터베이스 저장 구조 이해

인덱스 생성이 끝나면 같은 원문 chunk가 네 가지 로컬 산출물로 나뉘어 저장됩니다. 이 구조를 보면 RAG가 문서를 단순히 파일로 보관하는 것이 아니라, 답변에 필요한 검색 경로별 데이터베이스를 동시에 만든다는 점을 확인할 수 있습니다.

| 산출물 | 역할 |
| --- | --- |
| `chunks.jsonl` | 원문 조각의 표준 payload 저장소입니다. 모든 검색 채널이 다시 참조할 `chunk_id`, 본문, 원본 파일명, page/sheet/row, modality, tier 정보가 들어갑니다. |
| `qdrant` | dense text vector, dense image vector, sparse vector, chunk payload를 함께 저장하는 local Qdrant 저장소입니다. 의미 기반 검색과 이미지 modality 검색의 기준점입니다. |
| `bm25.json` | chunk별 tokenized keyword 문서를 저장합니다. 고유 명칭, 숫자, 표기 그대로의 키워드가 중요한 질문에 유리합니다. |
| `graph/graph_index.json` | entity, relations, entity_to_chunks, entity_neighbors, chunk payload를 저장합니다. 기관/위성/계약처럼 관계를 따라가야 하는 질문에 유리합니다. |

아래 셀은 산출물 존재 여부와 파일 수, chunk payload 샘플, modality 분포, BM25 token 문서, graph schema, Qdrant local collection 정보를 한국어 표와 JSON preview로 출력합니다. 또한 `graph_index.json`의 entity, relation, entity_to_chunks를 노드와 엣지 형태로 시각화해 지식 데이터베이스가 어떤 연결 구조를 갖는지 보여줍니다.

In [ ]:
import html as html_lib
from collections import Counter


def _artifact_file_count(path):
    if path.is_dir():
        return sum(1 for child in path.rglob("*") if child.is_file())
    return 1 if path.exists() else 0


def _short_text(value, max_chars=220):
    text = " ".join(str(value or "").split())
    return text if len(text) <= max_chars else text[: max_chars - 1].rstrip() + "..."


def _knowledge_graph_html(graph_payload, chunk_payloads):
    entity_to_chunks = dict(graph_payload.get("entity_to_chunks") or {})
    entity_text = dict(graph_payload.get("entity_text") or {})
    entity_types = dict(graph_payload.get("entity_types") or {})
    chunk_lookup = dict(graph_payload.get("chunks") or {})
    if not chunk_lookup:
        chunk_lookup = {payload.get("chunk_id"): payload for payload in chunk_payloads if payload.get("chunk_id")}

    entity_nodes = []
    chunk_nodes = []
    entity_chunk_edges = []
    seen_chunks = set()
    for entity_id, chunk_ids in list(entity_to_chunks.items())[:6]:
        label = entity_text.get(entity_id, entity_id)
        entity_nodes.append(
            {
                "id": entity_id,
                "label": _short_text(label, 28),
                "type": entity_types.get(entity_id, "Entity"),
                "chunk_count": len(chunk_ids),
            }
        )
        for chunk_id in chunk_ids[:2]:
            payload = chunk_lookup.get(chunk_id, {})
            if chunk_id not in seen_chunks:
                seen_chunks.add(chunk_id)
                chunk_nodes.append(
                    {
                        "id": chunk_id,
                        "label": _short_text(payload.get("source_file") or chunk_id, 34),
                        "modality": payload.get("modality", "chunk"),
                    }
                )
            entity_chunk_edges.append((entity_id, chunk_id))
            if len(chunk_nodes) >= 6:
                break
        if len(chunk_nodes) >= 6:
            break

    relation_edges = []
    for relation in list(graph_payload.get("relations") or [])[:6]:
        source = str(relation.get("source") or "")
        target = str(relation.get("target") or "")
        rel_type = str(relation.get("type") or "RELATED_TO")
        if source and target:
            relation_edges.append((source, rel_type, target))

    def node_html(node, kind):
        title = html_lib.escape(str(node.get("label") or node.get("id") or "node"))
        meta = html_lib.escape(str(node.get("type") or node.get("modality") or kind))
        count = node.get("chunk_count")
        if count is not None:
            meta += f" · chunks {html_lib.escape(str(count))}"
        return (
            f"<div class='kg-node kg-{kind}'>"
            f"<div class='kg-node-title'>{title}</div>"
            f"<div class='kg-node-meta'>{meta}</div>"
            "</div>"
        )

    artifact_nodes = [
        {"label": "chunks.jsonl", "type": "표준 chunk payload"},
        {"label": "qdrant", "type": "dense/sparse vector + payload"},
        {"label": "bm25.json", "type": "tokenized keyword documents"},
        {"label": "graph_index.json", "type": "entity / relation / evidence"},
    ]
    entity_html = "".join(node_html(node, "entity") for node in entity_nodes) or "<div class='kg-empty'>entity 노드가 없습니다.</div>"
    chunk_html = "".join(node_html(node, "chunk") for node in chunk_nodes) or "<div class='kg-empty'>chunk 노드가 없습니다.</div>"
    artifact_html = "".join(node_html(node, "artifact") for node in artifact_nodes)
    entity_edge_html = "".join(
        "<div class='kg-edge'>"
        + html_lib.escape(entity_text.get(entity_id, entity_id))
        + " <span>entity_to_chunks</span> "
        + html_lib.escape(_short_text(chunk_id, 42))
        + "</div>"
        for entity_id, chunk_id in entity_chunk_edges[:8]
    ) or "<div class='kg-empty'>entity_to_chunks edge가 없습니다.</div>"
    relation_edge_html = "".join(
        "<div class='kg-edge kg-relation-edge'>"
        + html_lib.escape(entity_text.get(source, source))
        + " <span>" + html_lib.escape(rel_type) + "</span> "
        + html_lib.escape(entity_text.get(target, target))
        + "</div>"
        for source, rel_type, target in relation_edges
    ) or "<div class='kg-empty'>relation edge가 없습니다.</div>"

    return f"""
<style>
.kg-wrap {{ font-family: Arial, sans-serif; border:1px solid #d0d7de; border-radius:8px; padding:14px; margin:10px 0 18px; background:#fff; }}
.kg-title {{ font-weight:700; font-size:16px; color:#24292f; margin-bottom:4px; }}
.kg-subtitle {{ color:#57606a; font-size:13px; margin-bottom:12px; }}
.kg-layer {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap:10px; margin:10px 0; }}
.kg-node {{ border:1px solid #d0d7de; border-radius:999px; padding:10px 12px; min-height:46px; background:#f6f8fa; box-shadow:0 1px 2px rgba(31,35,40,.06); }}
.kg-artifact {{ background:#eef6ff; border-color:#9ec5fe; }}
.kg-entity {{ background:#dafbe1; border-color:#8ddb8c; }}
.kg-chunk {{ background:#fff8c5; border-color:#eac54f; }}
.kg-node-title {{ font-weight:700; color:#24292f; font-size:13px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
.kg-node-meta {{ color:#57606a; font-size:12px; margin-top:3px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
.kg-edge-panel {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap:10px; margin:8px 0; }}
.kg-edge-box {{ border-left:3px solid #8c959f; background:#f6f8fa; padding:8px 10px; }}
.kg-edge-box-title {{ font-weight:700; font-size:12px; color:#57606a; margin-bottom:6px; }}
.kg-edge {{ font-size:12px; color:#24292f; margin:4px 0; }}
.kg-edge span {{ color:#8250df; font-weight:700; }}
.kg-empty {{ color:#6e7781; font-size:12px; }}
</style>
<div class='kg-wrap'>
  <div class='kg-title'>지식 데이터베이스 노드 보기</div>
  <div class='kg-subtitle'>인덱스 산출물, entity 노드, chunk 노드, relation edge가 어떻게 연결되는지 `graph_index.json` 샘플로 확인합니다.</div>
  <div class='kg-layer'>{artifact_html}</div>
  <div class='kg-edge-panel'>
    <div class='kg-edge-box'><div class='kg-edge-box-title'>entity_to_chunks edge</div>{entity_edge_html}</div>
    <div class='kg-edge-box'><div class='kg-edge-box-title'>relation edge</div>{relation_edge_html}</div>
  </div>
  <div class='kg-layer'>{entity_html}</div>
  <div class='kg-layer'>{chunk_html}</div>
</div>
"""


def _html_table(rows, columns):
    header = "".join(
        "<th style='padding:8px; border-bottom:1px solid #d0d7de; text-align:left; background:#f6f8fa;'>"
        + html_lib.escape(column)
        + "</th>"
        for column in columns
    )
    body = []
    for row in rows:
        cells = "".join(
            "<td style='padding:8px; border-bottom:1px solid #eaecf0; vertical-align:top;'>"
            + html_lib.escape(str(row.get(column, "-")))
            + "</td>"
            for column in columns
        )
        body.append("<tr>" + cells + "</tr>")
    return (
        "<div style='overflow-x:auto; margin:8px 0 16px;'>"
        "<table style='border-collapse:collapse; width:100%; font-size:14px; line-height:1.5;'>"
        f"<thead><tr>{header}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )


chunks_path = INDEX_ARTIFACTS["chunks"]
bm25_path = INDEX_ARTIFACTS["bm25"]
graph_path = INDEX_ARTIFACTS["graph_index"]
qdrant_path = INDEX_ARTIFACTS["qdrant"]

storage_rows = [
    {
        "산출물": "chunks.jsonl",
        "역할": "원문 조각의 표준 payload 저장소",
        "주요 저장 내용": "chunk_id, text, source_file, modality, page/sheet/row, tier, asset refs",
        "존재": chunks_path.exists(),
        "파일 수": _artifact_file_count(chunks_path),
        "경로": str(chunks_path),
    },
    {
        "산출물": "qdrant",
        "역할": "dense text/image vector, sparse vector, payload 저장소",
        "주요 저장 내용": f"collection={BUILD_REPORT.get('qdrant_collection', 'unknown')}, point payload는 chunks.jsonl payload와 같은 기준",
        "존재": qdrant_path.exists(),
        "파일 수": _artifact_file_count(qdrant_path),
        "경로": str(qdrant_path),
    },
    {
        "산출물": "bm25.json",
        "역할": "chunk별 tokenized keyword 문서 저장소",
        "주요 저장 내용": "chunk_ids, documents[token list]",
        "존재": bm25_path.exists(),
        "파일 수": _artifact_file_count(bm25_path),
        "경로": str(bm25_path),
    },
    {
        "산출물": "graph/graph_index.json",
        "역할": "entity, relations, entity_to_chunks, neighbor index 저장소",
        "주요 저장 내용": "entity_text, entity_types, relations, entity_to_chunks, entity_neighbors, chunks",
        "존재": graph_path.exists(),
        "파일 수": _artifact_file_count(graph_path),
        "경로": str(graph_path),
    },
]
display(HTML(_html_table(storage_rows, ["산출물", "역할", "주요 저장 내용", "존재", "파일 수", "경로"])))

chunk_payloads = []
if chunks_path.exists():
    with chunks_path.open("r", encoding="utf-8") as f:
        chunk_payloads = [json.loads(line) for line in f if line.strip()]

chunk_preview = []
for payload in chunk_payloads[:2]:
    chunk_preview.append(
        {
            "chunk_id": payload.get("chunk_id"),
            "source_file": payload.get("source_file"),
            "modality": payload.get("modality"),
            "page": payload.get("page"),
            "sheet": payload.get("sheet"),
            "row": payload.get("row"),
            "tier": payload.get("tier"),
            "text_preview": _short_text(payload.get("text")),
        }
    )
modality_counts = dict(Counter(str(payload.get("modality") or "unknown") for payload in chunk_payloads))

bm25_payload = json.loads(bm25_path.read_text(encoding="utf-8")) if bm25_path.exists() else {}
bm25_documents = list(bm25_payload.get("documents") or [])
bm25_doc_lengths = [len(doc) for doc in bm25_documents]
bm25_summary = {
    "문서 수": len(bm25_documents),
    "평균 토큰 길이": round(sum(bm25_doc_lengths) / max(1, len(bm25_doc_lengths)), 2),
    "샘플 token": bm25_documents[0][:12] if bm25_documents else [],
}

graph_payload = json.loads(graph_path.read_text(encoding="utf-8")) if graph_path.exists() else {}
entity_to_chunks = dict(graph_payload.get("entity_to_chunks") or {})
entity_text = dict(graph_payload.get("entity_text") or {})
entity_to_chunks_sample = [
    {
        "entity_id": entity_id,
        "entity_label": entity_text.get(entity_id, entity_id),
        "chunks": chunk_ids[:3],
    }
    for entity_id, chunk_ids in list(entity_to_chunks.items())[:3]
]
graph_summary = {
    "schema_version": graph_payload.get("schema_version"),
    "entity 수": len(entity_to_chunks),
    "relation 수": len(graph_payload.get("relations") or []),
    "entity_to_chunks 샘플": entity_to_chunks_sample,
}

KNOWLEDGE_GRAPH_HTML = _knowledge_graph_html(graph_payload, chunk_payloads)
display(HTML(KNOWLEDGE_GRAPH_HTML))

qdrant_json_path = qdrant_path / "vectors.json"
qdrant_debug_payload_sample = []
if qdrant_json_path.exists():
    qdrant_rows = json.loads(qdrant_json_path.read_text(encoding="utf-8"))
    for row in qdrant_rows[:2]:
        payload = dict(row.get("payload") or {})
        qdrant_debug_payload_sample.append(
            {
                "id": row.get("id"),
                "payload_keys": sorted(payload.keys())[:16],
                "chunk_id": payload.get("chunk_id"),
                "source_file": payload.get("source_file"),
            }
        )
qdrant_summary = {
    "collection_name": BUILD_REPORT.get("qdrant_collection"),
    "저장 경로": str(qdrant_path),
    "payload 설명": "Qdrant point payload는 chunks.jsonl의 표준 payload를 그대로 담아 검색 결과가 원문 위치와 source로 돌아갈 수 있게 합니다.",
    "로컬 파일 수": _artifact_file_count(qdrant_path),
    "json_debug_payload_sample": qdrant_debug_payload_sample,
}

DATABASE_PREVIEW = {
    "chunks.jsonl": {
        "chunk 수": len(chunk_payloads),
        "modality별 chunk 수": modality_counts,
        "payload 샘플 2개": chunk_preview,
    },
    "bm25.json": bm25_summary,
    "graph_index.json": graph_summary,
    "qdrant": qdrant_summary,
}

display(Markdown("### 저장 구조 JSON preview\n\n```json\n" + json.dumps(DATABASE_PREVIEW, ensure_ascii=False, indent=2) + "\n```"))

## 8B. Qdrant / Graph 저장 구조 시각화

이 셀은 8번 인덱스 생성 이후 만들어진 로컬 RAG 저장소가 실제로 어떤 파일과 데이터 구조로 구성되는지 확인합니다.

- Qdrant collection의 dense/sparse vector 설정과 point payload 샘플을 확인합니다.
- `chunks.jsonl`, `bm25.json`, `graph/graph_index.json`의 역할, 파일 수, 크기를 비교합니다.
- graph index의 entity, relation, entity_to_chunks 연결을 SVG 맵으로 시각화합니다.

인덱스 생성 셀을 실행한 뒤 이 셀을 실행하세요.

In [ ]:
# Qdrant point + graph_index ?? ?? ???
from aerospace_rag.notebook_runtime import format_storage_visualization

for section in format_storage_visualization(INDEX_DIR, build_report=BUILD_REPORT):
    if section["type"] == "markdown":
        display(Markdown(section["content"]))
    else:
        display(HTML(section["content"]))


## 9. 검색 단독 검증

LLM을 호출하기 전에 retrieval만 실행합니다. 답변 품질 문제가 생겼을 때 검색 문제인지 생성 문제인지 분리해서 볼 수 있습니다.

검색 채널은 질문 성격에 따라 서로 다른 강점을 가집니다.
- dense(Qdrant dense text/image): 표현이 달라도 의미가 가까운 문장, 표 설명, 이미지 설명을 찾는 데 유리합니다.
- sparse(BM25/Qdrant sparse): 위성명, 기관명, 날짜, 금액, 약어처럼 표면 키워드가 중요한 질문에 유리합니다.
- graph(graph-lite): 엔티티와 관계를 따라가야 하는 계약, 기관, 위성, 원인-결과 질문에 유리합니다.

진단 정보에서는 `channel_weights`가 채널별 결합 가중치, `query_segment`가 질문 유형 분류, `top_doc_channel_contributions`가 상위 chunk 점수에 각 채널이 기여한 값을 뜻합니다.

In [ ]:
from aerospace_rag.config import Settings
from aerospace_rag.notebook_runtime import format_retrieval_markdown
from aerospace_rag.stores.local_index import LocalIndex

RETRIEVAL_QUESTION = "위성영상 가격은 저장영상과 신규촬영에서 어떻게 다른가?"
index = LocalIndex(INDEX_DIR, settings=Settings.from_env())
hits = index.search(RETRIEVAL_QUESTION, top_k=TOP_K)
display(Markdown(format_retrieval_markdown(RETRIEVAL_QUESTION, hits, index.last_diagnostics)))

## 10. LLM 답변 생성

동일 질문을 `ask()` 파이프라인으로 실행합니다. 모델 호출이 실패하면 명확한 오류가 나며, LLM 없이 확인하려면 4번 설정 셀에서 `ANSWER_PROVIDER = "extractive"`, `EXTRACTOR_LLM_BACKEND = "local_fallback"`로 바꿉니다. diagnostics에서 provider, retrieval 채널, weighted RRF 결합 상태를 함께 확인합니다.

In [ ]:
from aerospace_rag.notebook_runtime import format_answer_markdown
from aerospace_rag.pipeline import ask

question = RETRIEVAL_QUESTION
response = ask(question, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
display(Markdown(format_answer_markdown(response)))

## 11. 근거 확인

최종 답변에 사용된 source reference를 확인합니다. page/sheet/row와 channel score가 함께 출력됩니다.

In [ ]:
from aerospace_rag.notebook_runtime import format_sources_markdown

display(Markdown(format_sources_markdown(response.sources)))

## 12. 반복 질문 예시

여러 질문을 같은 index에 대해 실행해 retrieval/answer 흐름이 반복 가능하게 동작하는지 확인합니다.

In [ ]:
from aerospace_rag.notebook_runtime import build_response_row, format_results_table

sample_questions = [
    "H3 8호기 발사 실패 원인은?",
    "NASA solar sail 연구의 목적은?",
    "국가별 우주항공 예산과 인력 현황은?",
    "인공위성 판매대행사 선정 절차는 얼마나 걸리는가?",
]

SAMPLE_RESULTS = []
for q in sample_questions:
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    SAMPLE_RESULTS.append(build_response_row(q, r))

display(HTML(format_results_table(SAMPLE_RESULTS, columns=["question", "summary", "top_source", "source_count", "provider"])))

## 13. 실제 업무파일 RAG 검증

실제 `data/` 업무파일 전체로 만든 index에 대해 대표 질문을 실행합니다. 각 질문마다 provider, 검색 채널, top source, 고유 source 목록, 답변 preview를 출력해 Colab에서도 ingest부터 답변 생성까지 한 번에 검증할 수 있습니다.

In [ ]:
ACTUAL_RAG_QUESTIONS = [
    "H3 8호기 발사 경과의 핵심 내용은 무엇인가?",
    "NASA가 Momentus에 수여한 계약은 무엇인가?",
    "위성영상 가격에서 저장영상과 신규촬영은 어떻게 다른가?",
    "해외정부 우주항공 현황 문서에서 확인되는 주요 국가나 기관은 무엇인가?",
    "인공위성 질문응답 문서 기준으로 인공위성 관련 핵심 설명은 무엇인가?",
]

ACTUAL_RAG_RESULTS = []
for case_idx, q in enumerate(ACTUAL_RAG_QUESTIONS, start=1):
    r = ask(q, index_dir=INDEX_DIR, top_k=TOP_K, provider=ANSWER_PROVIDER, debug=True)
    ACTUAL_RAG_RESULTS.append(build_response_row(q, r, case=case_idx))

display(HTML(format_results_table(ACTUAL_RAG_RESULTS, columns=["case", "question", "summary", "top_source", "source_count", "channels", "source_files"])))